In [19]:
!pip install -U transformers

In [20]:
import transformers
print(transformers.__version__)

5.8.0


In [1]:
# pip install --upgrade transformers

In [2]:
!pip install evaluate

In [3]:
import evaluate


In [4]:
# train_bert.py

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
# import evaluate
import numpy as np

In [5]:
dataset = load_dataset("imdb")

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [6]:
print(dataset["train"][0])


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [7]:
print(dataset["train"][0]["text"])

I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, eve

In [8]:
print(dataset["train"][0]["label"])

0


In [9]:
print(unique_labels := dataset["train"].unique("label"))

[0, 1]


## Step 5 — Load BERT Tokenizer

In [10]:
checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

## ✂️ Step 6 — Tokenize Data

BERT does NOT read raw text directly.

It reads tokens.

### create tokenization function

In [11]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

### Apply to Dataset

In [12]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

### Step 7 — Prepare PyTorch Format

In [13]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")

tokenized_dataset.set_format("torch")

### Load Pretrained BERT

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
accuracy = evaluate.load("accuracy")

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

## Step 10 — Training Configuration

In [21]:
training_args = TrainingArguments(
    output_dir="./results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=100
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
from transformers import TrainingArguments

Step 11 — Create Trainer

In [23]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["test"],

    processing_class=tokenizer,

    compute_metrics=compute_metrics
)

In [24]:
tokenizer=tokenizer

## Step 12 — Start Training

In [25]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.266201,0.256332,0.913480
2,0.164430,0.326697,0.922600


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=6250, training_loss=0.23941881927490236, metrics={'train_runtime': 3402.9181, 'train_samples_per_second': 14.693, 'train_steps_per_second': 1.837, 'total_flos': 6577776384000000.0, 'train_loss': 0.23941881927490236, 'epoch': 2.0})

## evaluate model

In [26]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy
0.164430,0.326697,2,0.922600


{'eval_loss': 0.32669734954833984, 'eval_accuracy': 0.9226}

# Save Properly

In [27]:
trainer.save_model("./bert-sentiment-model")

tokenizer.save_pretrained("./bert-sentiment-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./bert-sentiment-model/tokenizer_config.json',
 './bert-sentiment-model/tokenizer.json')

# How to Load Saved Model Later

## ✅ Load Model

In [28]:
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification

## Load Tokenizer

In [29]:
tokenizer = AutoTokenizer.from_pretrained(
    "./bert-sentiment-model"
)

## Load Model

In [30]:
model = AutoModelForSequenceClassification.from_pretrained(
    "./bert-sentiment-model"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

# 🔮 Predict New Text

In [31]:
text = "This movie was fantastic!"

## Convert Text → Tokens

In [32]:
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

## Run Prediction

In [33]:
import torch

with torch.no_grad():
    outputs = model(**inputs)

In [34]:
prediction = torch.argmax(
    outputs.logits,
    dim=-1
)

print(prediction.item())

1


In [36]:
if prediction.item() == 1:
  print("Positive")
else:
    print("Negative")



Positive
